# Assignment 3 CPSC 542

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torchvision import models
from torchvision.transforms import v2
from tqdm import tqdm
import kagglehub
from torchvision.io import read_image
from torchvision.tv_tensors import BoundingBoxes
import cv2
from ensemble_boxes import weighted_boxes_fusion



/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
print(np.__version__)

2.2.6


In [23]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("awsaf49/vinbigdata-1024-image-dataset")

print("Path to dataset files:", path)

Resuming download from 274726912 bytes (8466603624 bytes left)...
Resuming download to /root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/1.archive (274726912/8741330536) bytes left.


100%|██████████| 8.14G/8.14G [02:13<00:00, 63.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/versions/1


* Note: We don't want to use the test set here because that is part of the competition scoring, we will make our own train/val/test splits using the testing set

In [2]:
records = pd.read_csv("vinbigdata-1024-image-dataset/versions/1/vinbigdata/train.csv")


In [3]:
records.head()


,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max,width,height
0,50a418190bc3fb1ef1633bf9678929b3,No finding,14,R11,NaN,NaN,NaN,NaN,2332,2580
1,21a10246a5ec7af151081d0cd6d65dc9,No finding,14,R7,NaN,NaN,NaN,NaN,2954,3159
2,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,691.0,1375.0,1653.0,1831.0,2080,2336
3,051132a778e61a86eb147c7c6f564dfe,Aortic enlargement,0,R10,1264.0,743.0,1611.0,1019.0,2304,2880
4,063319de25ce7edb9b1c6b8881290140,No finding,14,R10,NaN,NaN,NaN,NaN,2540,3072


In [4]:
records.isna().sum()

image_id          0
class_name        0
class_id          0
rad_id            0
x_min         31818
y_min         31818
x_max         31818
y_max         31818
width             0
height            0
dtype: int64

## EDA and Pipeline for New Dataset

Handle Missing or Impossible Values

In [5]:
records.shape


(67914, 10)

### Consolidate Images based on rad_id

We have 67,914 rows in the training set because mutliple radiologists have annotated each image, rad_id tells us which radiologist did the annotation, so we will use Weighted Box Fusion to combine xrays with annotations from different radiologists on the same x-ray into one solid set.

In [6]:
def consolidate_dataset(df, iou_thr=0.5, skip_box_thr=0.0001):
    # Separate 'No Finding' (Class 14) 
    findings_df = df[df['class_id'] != 14].copy()
    
    # Pre-calculate mapping for class_id -> class_name to keep it fast
    class_map = findings_df[['class_id', 'class_name']].drop_duplicates().set_index('class_id')['class_name'].to_dict()
    # Pre-calculate mapping for image_id -> (width, height)
    dim_map = df[['image_id', 'width', 'height']].drop_duplicates().set_index('image_id').to_dict('index')
    
    # Normalize coordinates for WBF [0, 1]
    findings_df['x_min'] /= findings_df['width']
    findings_df['y_min'] /= findings_df['height']
    findings_df['x_max'] /= findings_df['width']
    findings_df['y_max'] /= findings_df['height']
    
    new_rows = []
    image_ids = findings_df['image_id'].unique()
    
    for img_id in tqdm(image_ids, desc="Fusing Boxes"):
        img_group = findings_df[findings_df['image_id'] == img_id]
        
        boxes_list = [img_group[['x_min', 'y_min', 'x_max', 'y_max']].values.tolist()]
        scores_list = [[1.0] * len(img_group)] 
        labels_list = [img_group['class_id'].values.tolist()]
        
        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list, 
            weights=None, iou_thr=iou_thr, skip_box_thr=skip_box_thr
        )
        
        # Get dimensions once for this image
        w = dim_map[img_id]['width']
        h = dim_map[img_id]['height']
        
        for i in range(len(boxes)):
            cls_id = int(labels[i])
            new_rows.append({
                'image_id': img_id,
                'class_id': cls_id,
                'class_name': class_map[cls_id],
                'width': w,
                'height': h,
                'x_min': boxes[i][0],
                'y_min': boxes[i][1],
                'x_max': boxes[i][2],
                'y_max': boxes[i][3]
            })
            
    return pd.DataFrame(new_rows)

consolidated_records = consolidate_dataset(records)

Fusing Boxes:   0%|          | 0/4394 [00:00<?, ?it/s]

Fusing Boxes: 100%|██████████| 4394/4394 [00:15<00:00, 290.19it/s]


Re-introduce No Finding and change the NaN bounding boxes to "0 0 1 1" as specified on the kaggle site. We want No Finding to have a confidence of 1 with that 1 pixel bounding box.

In [ ]:

no_findings_unique = records[records['class_id'] == 14].drop_duplicates(subset='image_id').copy()

# Clean up the coordinates for No Finding (using the Kaggle 1x1 pixel convention)
no_findings_unique[['x_min', 'y_min', 'x_max', 'y_max']] = [0, 0, 1, 1] 

# Combine them
final_records = pd.concat([consolidated_records, no_findings_unique], ignore_index=True)
final_records = final_records.drop(columns=['rad_id'])

In [8]:
final_records.head()

,image_id,class_id,class_name,width,height,x_min,y_min,x_max,y_max
0,9a5094b2563a1ef3ff50dc5c7ff71345,0,Aortic enlargement,2080,2336,0.505769,0.306079,0.624519,0.413527
1,9a5094b2563a1ef3ff50dc5c7ff71345,11,Pleural thickening,2080,2336,0.860096,0.740154,0.901442,0.852740
2,9a5094b2563a1ef3ff50dc5c7ff71345,10,Pleural effusion,2080,2336,0.860096,0.740154,0.901442,0.852740
3,9a5094b2563a1ef3ff50dc5c7ff71345,3,Cardiomegaly,2080,2336,0.332051,0.579766,0.797436,0.769549
4,051132a778e61a86eb147c7c6f564dfe,11,Pleural thickening,2304,2880,0.690972,0.156944,0.782986,0.209722


In [9]:
final_records.shape

(34540, 9)

In [10]:
final_records.isna().sum()

image_id      0
class_id      0
class_name    0
width         0
height        0
x_min         0
y_min         0
x_max         0
y_max         0
dtype: int64

Now we have a cleaned dataset with "combined" radiologist bounding boxes, no missing values.

Check for how many unique images there are since we have repeats

In [11]:
print("Total number of patients: ", records['image_id'].nunique(), "\n")

Total number of patients:  15000 



We have 15,000 unique images

### Data Augmentation

In [24]:
import os

# 1. The ID that failed
problem_id = "d3ef9ae515fb3cd3565e2c9875b3e0aa"
IMAGE_DIRECTORY = "/root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/versions/1/vinbigdata/train"

# 2. Check for hidden characters in the ID (like spaces)
print(f"Original ID length: {len(problem_id)}")
clean_id = problem_id.strip()
print(f"Cleaned ID length: {len(clean_id)}")

# 3. Construct the path manually
constructed_path = os.path.join(IMAGE_DIRECTORY, f"{clean_id}.png")
print(f"Full path we are testing: '{constructed_path}'")

# 4. Check if that specific file exists
exists = os.path.isfile(constructed_path)
print(f"File exists at that exact path: {exists}")

# 5. List the first few actual filenames on disk to check extensions/case
actual_files = os.listdir(IMAGE_DIRECTORY)
print(f"First 5 actual files on disk: {actual_files[:5]}")

Original ID length: 32
Cleaned ID length: 32
Full path we are testing: '/root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/versions/1/vinbigdata/train/d3ef9ae515fb3cd3565e2c9875b3e0aa.png'
File exists at that exact path: True
First 5 actual files on disk: ['a04cd3322f452395867fa98d676e9486.png', 'd4d41e0e0e7c512d05cdbd89401ee7bf.png', 'f3185be657c553fac1498f0d8088d56d.png', '0f34355eca9e6fb0fabab21711f0a2ec.png', 'd0252f72f9db197d04d2f59c8f77f908.png']


In [ ]:
train_transforms = v2.Compose([
    v2.ToImage(),                          # Convert to tensor-friendly format
    v2.Grayscale(num_output_channels=3),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=10),
    v2.ColorJitter(brightness=0.2, contrast=0.2), 
    v2.Resize(size=(512, 512), antialias=True),
    v2.ToDtype(torch.float32, scale=True), # Scales 0-255 to 0-1
    # v2.Normalize(mean=[0.485], std=[0.229])
    # Normalize stays here for both models. RT-DETR has no internal normalization
    # so this is the only place it happens. Faster R-CNN normalizes internally
    # via GeneralizedRCNNTransform — we neuter that by passing image_mean=[0,0,0]
    # and image_std=[1,1,1] at model construction time (see get_fasterrcnn_model).
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

IMAGE_DIRECTORY = "/root/.cache/kagglehub/datasets/awsaf49/vinbigdata-1024-image-dataset/versions/1/vinbigdata/train"

class VinBigDataDataset(torch.utils.data.Dataset):
    def __init__(self, df, img_dir, transforms=None):
        self.df = df
        self.img_dir = img_dir
        self.transforms = transforms
        self.img_ids = df['image_id'].unique()

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_path = f"{self.img_dir}/{img_id}.png"
        
        img = read_image(img_path) 
        
        # Originally we passed all boxes straight to BoundingBoxes including the 0,0,1,1 dummy
        # box that Kaggle uses for No Finding images. Faster R-CNN tries to actually regress
        # against that box and crashes because it's a 1x1 pixel degenerate annotation.
        # We still want No Finding images in training — the model needs to see healthy chest
        # x-rays so it learns when NOT to fire — we just pass empty boxes instead so Faster
        # R-CNN gets the right signal without breaking.
        #
        # Old code:
        # final_records = self.df[self.df['image_id'] == img_id]
        # bboxes = final_records[['x_min', 'y_min', 'x_max', 'y_max']].values
        # labels = torch.tensor(final_records['class_id'].values, dtype=torch.int64)
        # bboxes = BoundingBoxes(bboxes, format="XYXY", canvas_size=img.shape[-2:])

        img_records = self.df[self.df['image_id'] == img_id]
        labels = torch.tensor(img_records['class_id'].values, dtype=torch.int64)

        if (labels == 14).all():
            bboxes = BoundingBoxes(torch.zeros((0, 4), dtype=torch.float32), format="XYXY", canvas_size=img.shape[-2:])
            labels = torch.zeros(0, dtype=torch.int64)
        else:
            h, w = img.shape[-2], img.shape[-1]
            bboxes = img_records[['x_min', 'y_min', 'x_max', 'y_max']].values * [w, h, w, h]
            bboxes = BoundingBoxes(bboxes, format="XYXY", canvas_size=img.shape[-2:])

        if self.transforms:
            img, bboxes, labels = self.transforms(img, bboxes, labels)

        # Labels are 0-indexed (0-13) out of this function — No Finding images
        # return empty tensors so class 14 never appears here.
        # collate_fn_fasterrcnn adds +1 to shift to 1-14 (background=0).
        # collate_fn_detr keeps 0-indexed as RT-DETR expects.
        target = {
            "boxes": bboxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        return img, target

In [ ]:
val_transforms = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=3),
    v2.Resize(size=(512, 512), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    # v2.Normalize(mean=[0.485], std=[0.229])
    # Same rationale as train_transforms above.
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### Train, Validation and Test Splits

In [27]:
files_on_disk = [f.replace('.png', '') for f in os.listdir(IMAGE_DIRECTORY) if f.endswith('.png')]

# 2. Filter your main dataframe to only include these images
final_records_filtered = final_records[final_records['image_id'].isin(files_on_disk)].copy()

print(f"Original rows: {len(final_records)}")
print(f"Rows after filtering for existing images: {len(final_records_filtered)}")
print(f"Unique images available: {final_records_filtered['image_id'].nunique()}")

Original rows: 34540
Rows after filtering for existing images: 34540
Unique images available: 15000


In [28]:
# Want to split based on unique images
unique_images = final_records['image_id'].unique()

train_val_ids, test_ids = train_test_split(unique_images, test_size=0.10, random_state=42)

train_ids, val_ids = train_test_split(train_val_ids, test_size=0.11, random_state=42)

print(f"Train images: {len(train_ids)} | Val images: {len(val_ids)} | Test images: {len(test_ids)}")

Train images: 12015 | Val images: 1485 | Test images: 1500


Apply Transforms to the splits

In [29]:
train_df = final_records[final_records['image_id'].isin(train_ids)]
val_df = final_records[final_records['image_id'].isin(val_ids)]
test_df = final_records[final_records['image_id'].isin(test_ids)]


Create teh datasets with mappings and transforms

In [30]:
# Training Dataset
train_dataset = VinBigDataDataset(
    df=train_df, 
    img_dir=IMAGE_DIRECTORY,   # <--- The Mapping Happens Here
    transforms=train_transforms
)

# Validation Dataset
val_dataset = VinBigDataDataset(
    df=val_df, 
    img_dir=IMAGE_DIRECTORY, 
    transforms=val_transforms
)

# Test Dataset
test_dataset = VinBigDataDataset(
    df=test_df, 
    img_dir=IMAGE_DIRECTORY, 
    transforms=val_transforms
)

### DataLoader

In [31]:
def collate_fn(batch):
    """
    Handles variable numbers of bounding boxes per image in a batch.
    """
    return tuple(zip(*batch))

In [ ]:

# Hopefully 4 is fine for batch size 
BATCH_SIZE = 4 

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn,
    pin_memory=True  
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn,
    pin_memory=True
)

Test Pipeline

In [ ]:
images, targets = next(iter(train_loader))

print(f"--- Data Pipeline Verification ---")
print(f"Batch Size: {len(images)}")
print(f"Image Tensor Shape: {images[0].shape}") # Should be [3, 1024, 1024]
print(f"Number of Boxes in first image: {len(targets[0]['boxes'])}")
print(f"Labels in first image: {targets[0]['labels']}")
print(f"Device: {'GPU Ready' if images[0].is_cuda == False else 'Already on GPU'}")

--- Data Pipeline Verification ---
Batch Size: 4
Image Tensor Shape: torch.Size([1, 1024, 1024])
Number of Boxes in first image: 3
Labels in first image: tensor([13, 13,  6])
Device: GPU Ready


## Object Detection Pipeline — Marissa Estramonte

### Utilities

In [ ]:
import random

def set_seed(seed=42):
    # Call this at the top of every scenario so results are reproducible
    # across runs. Without this, model weight init and data shuffling are
    # different every time and you can't compare scenarios fairly.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### Collate Functions

In [ ]:
import torch
from torch.utils.data import DataLoader

FRCNN_NUM_CLASSES = 15   # 14 pathology classes + background (label 0)
DETR_NUM_LABELS   = 14   # RT-DETR has no background class, labels 0-13


def collate_fn_fasterrcnn(batch):
    images, targets = zip(*batch)
    new_targets = []
    for img, t in zip(images, targets):
        h, w = img.shape[-2], img.shape[-1]
        boxes = torch.as_tensor(t["boxes"], dtype=torch.float32)
        if boxes.numel() > 0:
            # clamp to valid image bounds
            boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, w)
            boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, h)
        labels = t["labels"].long() + 1  # shift 0-13 -> 1-14, background stays 0
        new_targets.append({"boxes": boxes, "labels": labels, "image_id": t["image_id"]})
    return list(images), new_targets


def collate_fn_detr(batch):
    images, targets = zip(*batch)
    new_targets = []
    for img, t in zip(images, targets):
        h, w = img.shape[-2], img.shape[-1]
        boxes = torch.as_tensor(t["boxes"], dtype=torch.float32)
        labels = t["labels"].long()

        if boxes.numel() > 0:
            boxes_xyxy = boxes.clone()
            # clamp to valid image bounds before normalizing
            boxes_xyxy[:, [0, 2]] = boxes_xyxy[:, [0, 2]].clamp(0, w)
            boxes_xyxy[:, [1, 3]] = boxes_xyxy[:, [1, 3]].clamp(0, h)
            cx = (boxes_xyxy[:, 0] + boxes_xyxy[:, 2]) / 2 / w
            cy = (boxes_xyxy[:, 1] + boxes_xyxy[:, 3]) / 2 / h
            bw = (boxes_xyxy[:, 2] - boxes_xyxy[:, 0]) / w
            bh = (boxes_xyxy[:, 3] - boxes_xyxy[:, 1]) / h
            boxes_cxcywh = torch.stack([cx, cy, bw, bh], dim=1).clamp(0, 1)
        else:
            boxes_xyxy   = torch.zeros((0, 4), dtype=torch.float32)
            boxes_cxcywh = torch.zeros((0, 4), dtype=torch.float32)

        new_targets.append({
            "boxes":      boxes_cxcywh,  # normalized cxcywh for DETR forward
            "boxes_xyxy": boxes_xyxy,    # xyxy pixel for mAP eval
            "labels":     labels,
            "image_id":   t["image_id"],
        })
    return list(images), new_targets

### DataLoaders

In [ ]:
# BATCH_SIZE=8 works fine at 512x512 on a 40GB DGX GPU.
# If a smoke test OOMs, drop to 4.
BATCH_SIZE  = 8
NUM_WORKERS = 2

train_loader_frcnn = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=NUM_WORKERS, collate_fn=collate_fn_fasterrcnn, pin_memory=True)
val_loader_frcnn   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, collate_fn=collate_fn_fasterrcnn, pin_memory=True)
test_loader_frcnn  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, collate_fn=collate_fn_fasterrcnn, pin_memory=True)

train_loader_detr  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=NUM_WORKERS, collate_fn=collate_fn_detr, pin_memory=True)
val_loader_detr    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, collate_fn=collate_fn_detr, pin_memory=True)
test_loader_detr   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, collate_fn=collate_fn_detr, pin_memory=True)

print(f"Train batches (frcnn/detr): {len(train_loader_frcnn)}")
print(f"Val   batches (frcnn/detr): {len(val_loader_frcnn)}")
print(f"Test  batches (frcnn/detr): {len(test_loader_frcnn)}")

### Custom CNN Backbone

In [ ]:
import torch.nn as nn
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool
from torchvision.models.detection.backbone_utils import BackboneWithFPN


class CustomCNNBackbone(nn.Module):
    # 4-stage CNN; strides 4/8/16/32, channels 64/128/256/512
    def __init__(self):
        super().__init__()
        self.stage1 = nn.Sequential(
            nn.Conv2d(3,   64,  3, stride=2, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64,  64,  3, stride=2, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
        )
        self.stage2 = nn.Sequential(
            nn.Conv2d(64,  128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),           nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.stage3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),           nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.stage4 = nn.Sequential(
            nn.Conv2d(256, 512, 3, stride=2, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1),           nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        s1 = self.stage1(x)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)
        return {"stage1": s1, "stage2": s2, "stage3": s3, "stage4": s4}


def _wrap_custom_backbone_with_fpn(out_channels=256):
    backbone = CustomCNNBackbone()
    return BackboneWithFPN(
        backbone,
        return_layers={"stage1": "0", "stage2": "1", "stage3": "2", "stage4": "3"},
        in_channels_list=[64, 128, 256, 512],
        out_channels=out_channels,
        extra_blocks=LastLevelMaxPool(),
    )


# sanity check: verify FPN output shapes before training
_bb = _wrap_custom_backbone_with_fpn()
_dummy = torch.zeros(1, 3, 512, 512)
with torch.no_grad():
    _out = _bb(_dummy)
print("Custom backbone FPN keys:", list(_out.keys()))
for k, v in _out.items():
    print(f"  {k}: {v.shape}")
del _bb, _dummy, _out

### Model Definitions

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from transformers import AutoModelForObjectDetection, AutoConfig


def get_fasterrcnn_model(num_classes=FRCNN_NUM_CLASSES, pretrained=False, custom_backbone=False):
    if custom_backbone:
        backbone = _wrap_custom_backbone_with_fpn(out_channels=256)
        model = FasterRCNN(backbone, num_classes=num_classes)
    elif pretrained:
        model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
        in_features = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    else:
        model = fasterrcnn_resnet50_fpn(weights=None, num_classes=num_classes)

    # we normalize in transforms, so null out the internal GeneralizedRCNNTransform
    # normalization to avoid doing it twice
    model.transform.image_mean = [0.0, 0.0, 0.0]
    model.transform.image_std  = [1.0, 1.0, 1.0]
    return model


def get_detr_model(num_labels=DETR_NUM_LABELS, pretrained=False):
    if pretrained:
        model = AutoModelForObjectDetection.from_pretrained(
            "PekingU/rtdetr_r50vd",
            num_labels=num_labels,
            ignore_mismatched_sizes=True,
        )
    else:
        # load architecture only, with random weights
        cfg = AutoConfig.from_pretrained("PekingU/rtdetr_r50vd", num_labels=num_labels)
        model = AutoModelForObjectDetection.from_config(cfg)
    return model

### Training Infrastructure

In [ ]:
def build_detection_optimizer(model, phase, model_type="fasterrcnn", head_lr=1e-3):
    # phase 0: everything, single LR (from-scratch scenarios a and b)
    # phase 1: backbone frozen, head only
    # phase 2: unfreeze backbone at 10x lower LR than head
    if model_type == "fasterrcnn":
        backbone_params = lambda: list(model.backbone.parameters())
        if phase == 0:
            opt = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=head_lr, weight_decay=1e-4,
            )
        elif phase == 1:
            for p in model.backbone.parameters():
                p.requires_grad_(False)
            opt = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=head_lr, weight_decay=1e-4,
            )
        else:
            for p in model.backbone.parameters():
                p.requires_grad_(True)
            bp = backbone_params()
            bp_set = set(bp)
            hp = [p for p in model.parameters() if p not in bp_set]
            opt = torch.optim.AdamW([
                {"params": bp, "lr": head_lr / 10},
                {"params": hp, "lr": head_lr},
            ], weight_decay=1e-4)

    else:  # detr
        backbone_params = lambda: list(model.model.backbone.parameters())
        if phase == 0:
            opt = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=head_lr, weight_decay=1e-4,
            )
        elif phase == 1:
            for p in model.model.backbone.parameters():
                p.requires_grad_(False)
            opt = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=head_lr, weight_decay=1e-4,
            )
        else:
            for p in model.model.backbone.parameters():
                p.requires_grad_(True)
            bp = backbone_params()
            bp_set = set(bp)
            hp = [p for p in model.parameters() if p not in bp_set]
            opt = torch.optim.AdamW([
                {"params": bp, "lr": head_lr / 10},
                {"params": hp, "lr": head_lr},
            ], weight_decay=1e-4)

    return opt

In [ ]:
def run_epoch(model, loader, optimizer, device, scaler, train=True, model_type="fasterrcnn"):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for images, targets in loader:
        images = [img.to(device) for img in images]

        if model_type == "fasterrcnn":
            targets_dev = [
                {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()}
                for t in targets
            ]
            if train:
                with torch.cuda.amp.autocast():
                    loss_dict = model(images, targets_dev)
                    loss = sum(loss_dict.values())
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()
            else:
                with torch.no_grad(), torch.cuda.amp.autocast():
                    preds = model(images)
                for pred, tgt in zip(preds, targets_dev):
                    # drop background predictions (label 0) before evaluation
                    keep = pred["labels"] != 0
                    all_preds.append({
                        "boxes":  pred["boxes"][keep].cpu(),
                        "labels": pred["labels"][keep].cpu(),
                        "scores": pred["scores"][keep].cpu(),
                    })
                    all_targets.append({
                        "boxes":  tgt["boxes"].cpu(),
                        "labels": tgt["labels"].cpu(),
                    })

        else:  # RT-DETR
            pixel_values = torch.stack(images).to(device)
            h, w = pixel_values.shape[-2], pixel_values.shape[-1]

            if train:
                hf_labels = [
                    {"class_labels": t["labels"].to(device), "boxes": t["boxes"].to(device)}
                    for t in targets
                ]
                with torch.cuda.amp.autocast():
                    out  = model(pixel_values=pixel_values, labels=hf_labels)
                    loss = out.loss
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item()
            else:
                with torch.no_grad(), torch.cuda.amp.autocast():
                    out = model(pixel_values=pixel_values)
                logits     = out.logits       # (B, num_queries, num_labels+1)
                pred_boxes = out.pred_boxes   # (B, num_queries, 4) cxcywh normalized

                for i, tgt in enumerate(targets):
                    scores_all = logits[i].softmax(-1)[:, :-1]  # drop no-object
                    scores, labels = scores_all.max(-1)
                    boxes = pred_boxes[i]

                    # RT-DETR outputs a fixed number of queries; keep top-100 by confidence
                    if len(scores) > 100:
                        top_idx = scores.topk(100).indices
                        scores  = scores[top_idx]
                        labels  = labels[top_idx]
                        boxes   = boxes[top_idx]

                    # cxcywh normalized -> xyxy pixel
                    cx, cy, bw, bh = boxes[:,0]*w, boxes[:,1]*h, boxes[:,2]*w, boxes[:,3]*h
                    xyxy = torch.stack([cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2], dim=1)

                    all_preds.append({
                        "boxes":  xyxy.cpu(),
                        "labels": labels.cpu(),
                        "scores": scores.cpu(),
                    })
                    all_targets.append({
                        "boxes":  tgt["boxes_xyxy"].cpu(),
                        "labels": tgt["labels"].cpu(),
                    })

    if train:
        return total_loss / max(len(loader), 1)
    return all_preds, all_targets

In [ ]:
def evaluate_metrics(preds, targets, num_classes):
    from torchmetrics.detection import MeanAveragePrecision
    metric = MeanAveragePrecision(iou_thresholds=[0.5], class_metrics=True)
    metric.update(preds, targets)
    result  = metric.compute()
    map50   = result["map_50"].item()
    per_cls = result.get("map_per_class", None)
    return map50, per_cls


def save_results(scenario_name, config, metrics_history, best_per_cls=None, results_dir="results"):
    import csv, os, json as _json
    out = f"{results_dir}/{scenario_name}"
    os.makedirs(out, exist_ok=True)

    if best_per_cls is not None:
        config = {**config, "best_per_cls_map50": best_per_cls}
    with open(f"{out}/config.json", "w") as f:
        _json.dump(config, f, indent=2)

    if metrics_history:
        keys = list(metrics_history[0].keys())
        with open(f"{out}/metrics.csv", "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(metrics_history)

    print(f"Saved {scenario_name} -> {out}/")

### Scenario Training

In [ ]:
import os, time

PATIENCE = 5  # early stopping patience for all phases


def train_scenario(scenario, device, epochs_p0=20, epochs_p1=15, epochs_p2=25, head_lr=1e-3):
    # scenario a: custom CNN + Faster R-CNN, from scratch
    # scenario b: RT-DETR, from scratch
    # scenario c: Faster R-CNN pretrained, backbone frozen throughout
    # scenario d: RT-DETR pretrained, two-phase fine-tuning

    set_seed(42)
    t0 = time.time()

    cfgs = {
        "a": dict(model_type="fasterrcnn", pretrained=False, custom_backbone=True),
        "b": dict(model_type="detr",       pretrained=False, custom_backbone=False),
        "c": dict(model_type="fasterrcnn", pretrained=True,  custom_backbone=False),
        "d": dict(model_type="detr",       pretrained=True,  custom_backbone=False),
    }
    cfg        = cfgs[scenario]
    model_type = cfg["model_type"]

    if model_type == "fasterrcnn":
        model = get_fasterrcnn_model(
            num_classes=FRCNN_NUM_CLASSES,
            pretrained=cfg["pretrained"],
            custom_backbone=cfg["custom_backbone"],
        )
        train_loader, val_loader = train_loader_frcnn, val_loader_frcnn
        n_cls = FRCNN_NUM_CLASSES
    else:
        model = get_detr_model(num_labels=DETR_NUM_LABELS, pretrained=cfg["pretrained"])
        train_loader, val_loader = train_loader_detr, val_loader_detr
        n_cls = DETR_NUM_LABELS

    model  = model.to(device)
    scaler = torch.cuda.amp.GradScaler()

    metrics_history = []
    best_map     = 0.0
    best_per_cls = None
    best_state   = None

    # ----- Scenario A & B: from scratch (phase 0 only) -----
    if scenario in ("a", "b"):
        optimizer = build_detection_optimizer(model, phase=0, model_type=model_type, head_lr=head_lr)
        # ReduceLROnPlateau backs off LR when mAP plateaus before early stopping kicks in
        scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=3
        )
        no_improve = 0

        for epoch in range(epochs_p0):
            loss = run_epoch(model, train_loader, optimizer, device, scaler, train=True,  model_type=model_type)
            p, t = run_epoch(model, val_loader,   optimizer, device, scaler, train=False, model_type=model_type)
            map50, per_cls = evaluate_metrics(p, t, n_cls)
            scheduler.step(map50)

            metrics_history.append({"phase": 0, "epoch": epoch+1, "train_loss": loss, "val_map50": map50})
            print(f"[{scenario}] p0 epoch {epoch+1}/{epochs_p0}  loss={loss:.4f}  mAP@0.5={map50:.4f}")

            if map50 > best_map:
                best_map   = map50
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                best_per_cls = per_cls.tolist() if per_cls is not None else None
                no_improve   = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  early stop epoch {epoch+1}")
                break

    # ----- Scenario C: pretrained frozen (phase 1 only) -----
    elif scenario == "c":
        optimizer = build_detection_optimizer(model, phase=1, model_type=model_type, head_lr=head_lr)

        # belt-and-suspenders: confirm backbone is frozen at start of training
        assert all(not p.requires_grad for p in model.backbone.parameters()),             "scenario (c): backbone parameters should all be frozen at training start"

        no_improve = 0
        for epoch in range(epochs_p1):
            loss = run_epoch(model, train_loader, optimizer, device, scaler, train=True,  model_type=model_type)
            p, t = run_epoch(model, val_loader,   optimizer, device, scaler, train=False, model_type=model_type)
            map50, per_cls = evaluate_metrics(p, t, n_cls)

            metrics_history.append({"phase": 1, "epoch": epoch+1, "train_loss": loss, "val_map50": map50})
            print(f"[{scenario}] p1 epoch {epoch+1}/{epochs_p1}  loss={loss:.4f}  mAP@0.5={map50:.4f}")

            if map50 > best_map:
                best_map   = map50
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                best_per_cls = per_cls.tolist() if per_cls is not None else None
                no_improve   = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  early stop epoch {epoch+1}")
                break

        # confirm backbone stayed frozen throughout -- nothing in this scenario should have
        # touched requires_grad, but good to document for the write-up
        assert all(not p.requires_grad for p in model.backbone.parameters()),             "scenario (c): backbone parameters should still be frozen at end of training"

    # ----- Scenario D: pretrained fine-tuned (phase 1 -> phase 2) -----
    else:
        # phase 1: frozen backbone
        optimizer  = build_detection_optimizer(model, phase=1, model_type=model_type, head_lr=head_lr)
        no_improve = 0

        for epoch in range(epochs_p1):
            loss = run_epoch(model, train_loader, optimizer, device, scaler, train=True,  model_type=model_type)
            p, t = run_epoch(model, val_loader,   optimizer, device, scaler, train=False, model_type=model_type)
            map50, per_cls = evaluate_metrics(p, t, n_cls)

            metrics_history.append({"phase": 1, "epoch": epoch+1, "train_loss": loss, "val_map50": map50})
            print(f"[{scenario}] p1 epoch {epoch+1}/{epochs_p1}  loss={loss:.4f}  mAP@0.5={map50:.4f}")

            if map50 > best_map:
                best_map   = map50
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                best_per_cls = per_cls.tolist() if per_cls is not None else None
                no_improve   = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  early stop p1 epoch {epoch+1}")
                break

        # load best p1 weights before unfreezing backbone
        if best_state is not None:
            model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

        # phase 2: unfreeze backbone with differential LR
        optimizer  = build_detection_optimizer(model, phase=2, model_type=model_type, head_lr=head_lr)
        no_improve = 0
        p2_best    = 0.0   # track phase 2 improvement on its own trajectory

        for epoch in range(epochs_p2):
            loss = run_epoch(model, train_loader, optimizer, device, scaler, train=True,  model_type=model_type)
            p, t = run_epoch(model, val_loader,   optimizer, device, scaler, train=False, model_type=model_type)
            map50, per_cls = evaluate_metrics(p, t, n_cls)

            metrics_history.append({"phase": 2, "epoch": epoch+1, "train_loss": loss, "val_map50": map50})
            print(f"[{scenario}] p2 epoch {epoch+1}/{epochs_p2}  loss={loss:.4f}  mAP@0.5={map50:.4f}")

            if map50 > p2_best:
                p2_best    = map50
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                best_per_cls = per_cls.tolist() if per_cls is not None else None
                no_improve   = 0
            else:
                no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  early stop p2 epoch {epoch+1}")
                break

        best_map = max(best_map, p2_best)  # true overall best across both phases

    # save checkpoint and metrics
    out_dir = f"results/scenario_{scenario}"
    os.makedirs(out_dir, exist_ok=True)
    if best_state is not None:
        torch.save(best_state, f"{out_dir}/best_checkpoint.pt")

    wall_clock_seconds = round(time.time() - t0, 2)
    config = {
        "scenario": scenario, "model_type": model_type,
        "pretrained": cfg["pretrained"], "custom_backbone": cfg["custom_backbone"],
        "best_map50": best_map,
        "head_lr": head_lr, "batch_size": BATCH_SIZE, "image_size": 512,
        "epochs_p0": epochs_p0, "epochs_p1": epochs_p1, "epochs_p2": epochs_p2,
        "patience": PATIENCE, "wall_clock_seconds": wall_clock_seconds,
    }
    save_results(f"scenario_{scenario}", config, metrics_history, best_per_cls=best_per_cls)

    return model, metrics_history, best_map

### Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random as _random

CLASS_NAMES = [
    "Aortic enlargement", "Atelectasis", "Calcification", "Cardiomegaly",
    "Consolidation", "ILD", "Infiltration", "Lung Opacity",
    "Nodule/Mass", "Other lesion", "Pleural effusion", "Pleural thickening",
    "Pneumothorax", "Pulmonary fibrosis",
]


def visualize_predictions(model, dataset, device, model_type="fasterrcnn",
                           n=4, score_thresh=0.3, title=""):
    model.eval()
    indices = _random.sample(range(len(dataset)), k=n)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        img, target = dataset[idx]
        img_t = img.unsqueeze(0).to(device)

        with torch.no_grad():
            if model_type == "fasterrcnn":
                pred = model(img_t)[0]
                # filter background (label 0) and low-confidence
                keep = (pred["labels"] != 0) & (pred["scores"] >= score_thresh)
                pred_boxes  = pred["boxes"][keep].cpu()
                pred_labels = (pred["labels"][keep] - 1).cpu()  # back to 0-indexed for names
                pred_scores = pred["scores"][keep].cpu()
            else:
                out = model(pixel_values=img_t)
                h, w = img_t.shape[-2], img_t.shape[-1]
                scores_all = out.logits[0].softmax(-1)[:, :-1]
                scores, labels = scores_all.max(-1)
                boxes = out.pred_boxes[0]

                if len(scores) > 100:
                    top_idx = scores.topk(100).indices
                    scores  = scores[top_idx]
                    labels  = labels[top_idx]
                    boxes   = boxes[top_idx]

                keep = scores >= score_thresh
                scores = scores[keep]; labels = labels[keep]; boxes = boxes[keep]
                cx, cy, bw, bh = boxes[:,0]*w, boxes[:,1]*h, boxes[:,2]*w, boxes[:,3]*h
                pred_boxes  = torch.stack([cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2], dim=1).cpu()
                pred_labels = labels.cpu()
                pred_scores = scores.cpu()

        img_np = img.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        ax.imshow(img_np, cmap="gray")

        gt_boxes  = torch.as_tensor(target["boxes"],  dtype=torch.float32)
        gt_labels = torch.as_tensor(target["labels"], dtype=torch.long)
        for box, lbl in zip(gt_boxes, gt_labels):
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                            linewidth=1.5, edgecolor="lime", facecolor="none"))
            ax.text(x1, y1-2, CLASS_NAMES[int(lbl) % 14], fontsize=7, color="lime")

        for box, lbl, sc in zip(pred_boxes, pred_labels, pred_scores):
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                            linewidth=1.5, edgecolor="red", facecolor="none"))
            ax.text(x1, y2+8, f"{CLASS_NAMES[int(lbl) % 14]} {sc:.2f}", fontsize=7, color="red")

        ax.axis("off")
        ax.set_title(f"img {idx}")

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_scenario_comparison(results_dict):
    fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
    phase_colors = {0: "steelblue", 1: "darkorange", 2: "seagreen"}

    for ax, (sc, history) in zip(axes, results_dict.items()):
        by_phase = {}
        for row in history:
            by_phase.setdefault(row["phase"], []).append(row["val_map50"])

        offset = 0
        for ph, vals in sorted(by_phase.items()):
            xs = range(offset+1, offset+len(vals)+1)
            ax.plot(list(xs), vals, label=f"phase {ph}",
                    color=phase_colors.get(ph, "gray"), marker="o", ms=3)
            offset += len(vals)

        ax.set_title(f"Scenario ({sc})")
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=8)

    axes[0].set_ylabel("mAP@0.5")
    plt.suptitle("Validation mAP@0.5 — All Scenarios", fontsize=13)
    plt.tight_layout()
    plt.show()

### Dependency Check

In [ ]:
import importlib

REQUIRED = ["torch", "torchvision", "transformers", "torchmetrics",
            "numpy", "pandas", "matplotlib", "PIL"]

for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  MISSING  {pkg}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\ntorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Smoke Tests

In [ ]:
# Faster R-CNN (ResNet-50, from scratch) -- train and eval passes
_m = get_fasterrcnn_model(pretrained=False, custom_backbone=False).to(device)
_imgs = [torch.rand(3, 512, 512).to(device) for _ in range(2)]
_tgts = [
    {"boxes": torch.tensor([[10., 20., 100., 200.]], device=device),
     "labels": torch.tensor([1], device=device), "image_id": torch.tensor([0])},
    {"boxes": torch.tensor([[50., 60., 150., 250.]], device=device),
     "labels": torch.tensor([3], device=device), "image_id": torch.tensor([1])},
]
_m.train()
with torch.cuda.amp.autocast():
    _loss = sum(_m(_imgs, _tgts).values())
print(f"Faster R-CNN train loss: {_loss.item():.4f}")
_m.eval()
with torch.no_grad(), torch.cuda.amp.autocast():
    _preds = _m(_imgs)
print(f"Faster R-CNN eval boxes per image: {[len(p['boxes']) for p in _preds]}")
del _m, _imgs, _tgts, _preds

# RT-DETR (from scratch) -- train and eval passes
_md = get_detr_model(pretrained=False).to(device)
_imgs_d = torch.rand(2, 3, 512, 512).to(device)
_hf = [
    {"class_labels": torch.tensor([0], device=device),
     "boxes": torch.tensor([[0.5, 0.5, 0.2, 0.3]], device=device)},
    {"class_labels": torch.tensor([2], device=device),
     "boxes": torch.tensor([[0.3, 0.4, 0.1, 0.2]], device=device)},
]
_md.train()
with torch.cuda.amp.autocast():
    _out = _md(pixel_values=_imgs_d, labels=_hf)
print(f"RT-DETR train loss: {_out.loss.item():.4f}")
_md.eval()
with torch.no_grad(), torch.cuda.amp.autocast():
    _out_e = _md(pixel_values=_imgs_d)
print(f"RT-DETR eval logits shape: {_out_e.logits.shape}")
del _md, _imgs_d, _hf, _out, _out_e
print("Standard smoke test passed.")

In [ ]:
# Faster R-CNN with custom CNN backbone -- train and eval passes
_mc = get_fasterrcnn_model(pretrained=False, custom_backbone=True).to(device)
_imgs_c = [torch.rand(3, 512, 512).to(device) for _ in range(2)]
_tgts_c = [
    {"boxes": torch.tensor([[10., 20., 100., 200.]], device=device),
     "labels": torch.tensor([1], device=device), "image_id": torch.tensor([0])},
    {"boxes": torch.tensor([[50., 60., 150., 250.]], device=device),
     "labels": torch.tensor([3], device=device), "image_id": torch.tensor([1])},
]
_mc.train()
with torch.cuda.amp.autocast():
    _loss_c = sum(_mc(_imgs_c, _tgts_c).values())
print(f"Custom CNN backbone train loss: {_loss_c.item():.4f}")
_mc.eval()
with torch.no_grad(), torch.cuda.amp.autocast():
    _preds_c = _mc(_imgs_c)
print(f"Custom CNN backbone eval boxes per image: {[len(p['boxes']) for p in _preds_c]}")
del _mc, _imgs_c, _tgts_c, _preds_c
print("Custom backbone smoke test passed.")

### Run All Scenarios

In [ ]:
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results_all = {}

for s in ["a", "b", "c", "d"]:
    print(f"\n{'='*60}\nScenario ({s})\n{'='*60}")
    _model, _history, _best = train_scenario(s, device)
    results_all[s] = _history
    print(f"Scenario ({s}) best mAP@0.5: {_best:.4f}")
    del _model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll scenarios complete.")

In [ ]:
import json as _json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

scenario_meta_test = {
    "a": ("fasterrcnn", False, True),
    "b": ("detr",       False, False),
    "c": ("fasterrcnn", True,  False),
    "d": ("detr",       True,  False),
}

print(f"{'Scenario':<12}  {'Test mAP@0.5':>13}")
print("-" * 27)

for s, (mtype, pretrained, custom_bb) in scenario_meta_test.items():
    ckpt = f"results/scenario_{s}/best_checkpoint.pt"
    if not os.path.exists(ckpt):
        print(f"({s})            no checkpoint, skipping")
        continue

    if mtype == "fasterrcnn":
        _tm = get_fasterrcnn_model(pretrained=pretrained, custom_backbone=custom_bb).to(device)
        _tl, n_cls = test_loader_frcnn, FRCNN_NUM_CLASSES
    else:
        _tm = get_detr_model(pretrained=pretrained).to(device)
        _tl, n_cls = test_loader_detr, DETR_NUM_LABELS

    _tm.load_state_dict(torch.load(ckpt, map_location=device))
    p, t = run_epoch(_tm, _tl, None, device, None, train=False, model_type=mtype)
    test_map50, test_per_cls = evaluate_metrics(p, t, n_cls)

    result = {
        "scenario": s,
        "test_map50": test_map50,
        "test_per_cls_map50": test_per_cls.tolist() if test_per_cls is not None else None,
    }
    with open(f"results/scenario_{s}/test_results.json", "w") as f:
        _json.dump(result, f, indent=2)

    print(f"({s}){'':<10}  {test_map50:.4f}")
    del _tm

print("\nTest results saved to results/scenario_*/test_results.json")

In [ ]:
plot_scenario_comparison(results_all)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

scenario_meta = {
    "a": ("fasterrcnn", False, True,  "Custom CNN + Faster R-CNN (scratch)"),
    "b": ("detr",       False, False, "RT-DETR (scratch)"),
    "c": ("fasterrcnn", True,  False, "Faster R-CNN (pretrained, frozen backbone)"),
    "d": ("detr",       True,  False, "RT-DETR (pretrained, fine-tuned)"),
}

for s, (mtype, pretrained, custom_bb, label) in scenario_meta.items():
    ckpt = f"results/scenario_{s}/best_checkpoint.pt"
    if not os.path.exists(ckpt):
        print(f"No checkpoint for scenario ({s}), skipping.")
        continue

    if mtype == "fasterrcnn":
        _vm = get_fasterrcnn_model(pretrained=pretrained, custom_backbone=custom_bb).to(device)
    else:
        _vm = get_detr_model(pretrained=pretrained).to(device)

    _vm.load_state_dict(torch.load(ckpt, map_location=device))
    visualize_predictions(_vm, test_dataset, device, model_type=mtype,
                          n=4, score_thresh=0.3, title=f"Scenario ({s}): {label}")
    del _vm